# Stanowisko 4 - napęd omni (swerve drive) - kinematyka odwrotna

### Do zweryfikowania poprawności połączenia notatnika z robotem można wysłać na jedno z kół prędkości sterujące. 

Poniższe polecenie na temacie **/cmd_joint_states** opublikuje wiadomość typu **JointState** z pakietu **sensor_msgs**

W celu zatrzymania działającego programu w komórce **(oznaczenie * )** należy wcisnąć przycisk **interrupt the kernel (czarny kwadraty na górnym pasku)**


Po zweryfikowaniu prawidłowej komunikacji notatnika z robotem można na początku lini wstawić komentarz **#** , aby zablokować jej uruchamianie. Po wysłaniu polecenia jedno z kół powinno się kręcić.


In [ ]:
# ! lub %%bash umożliwia wywołanie w notatniku polecenia, które uruchamia się w konsoli (terminalu).

# Wysłanie prędkości na jedno z kół - jeśli dalsze uruchomienie nie jest potrzebne to wstawić znak komentarza #
# Ponisze polecenie powoduje uruchomienie nieskończonej pętli i uniemożliwia wywołanie kolejnych poleceń.
! ros2 topic pub -r 1 /cmd_joint_states sensor_msgs/msg/JointState "{velocity: [0.1, 0.0, 0.0, 0.0], position: [0.0, 0.0, 0.0, 0.0]}"
# ! ros2 topic pub -r 1 /cmd_joint_states sensor_msgs/msg/JointState "{velocity: [0.1, 0.0, 0.0, 0.0], position: [0.0, 0.0, 0.0, 0.0]}"

In [ ]:
# Zatrzymanie kół robota - wysłanie prędkości 0.0
# Jeśli dalsze uruchomienie nie jest potrzebne to wstawić znak komentarza #
# Ponisze polecenie powoduje uruchomienie nieskończonej pętli i uniemożliwia wywołanie kolejnych poleceń.
! ros2 topic pub -r 1 /cmd_joint_states sensor_msgs/msg/JointState "{velocity: [0.0, 0.0, 0.0, 0.0], position: [0.0, 0.0, 0.0, 0.0]}"

In [ ]:
# Opcjonalnie - wyświetlenie listy wszystkich widocznych tematów 
# To polecenie wykona się raz i nie uruchamia nieskończonej pętli. Można zakomentować, ale nie trzeba.
! ros2 topic list

---

## Import podstawowych bibliotek ROS2

In [ ]:
# Główna biblioteka ROS2 dla Pythona – pozwala na uruchamianie i obsługę nodów (programów w ROS2)
import rclpy 

# Klasa bazowa do tworzenia własnych nodów – każdy node w ROS2 dziedziczy po tej klasie
from rclpy.node import Node

# Import standardowej wiadomości JointState z pakietu sensor_msgs dla ROS2. Wiadomość służy do publikowania 
# i subskrybowania stanu przegubów (złącz) (np. pozycja, prędkość, moment obrotowy)
from sensor_msgs.msg import JointState

# Wiadomość ROS2 używana do przekazywania prędkości liniowych i kątowych robota (np. do sterowania ruchem)
from geometry_msgs.msg import Twist

## Import bibliotek do obsługi matematyki dla Python

In [ ]:
# Popularna biblioteka numeryczna w Pythonie – używana do obliczeń macierzowych i wektorowych
import numpy as np

# Standardowa biblioteka Pythona z funkcjami matematycznymi (np. sin, cos, sqrt)
import math

## <span style="color:red">Uzupełnij poniższą komórkę. (1/2)</span>

Wstaw parametry kinematyki wyznaczone w ćwiczeniu 1.

Parametry należy wykorzystać podczas dalszej implementacji wzorów.

---

In [ ]:
class Robot(Node):
    """
    Główna klasa programu, która obsługuje kinematykę odwrotną robota w ROS2. Kinematyka odwrotna polega 
    na przeliczeniu zadanych prędkości ruchu robota na prędkości obrotowe (i ewentualnie kąty skrętu) jego kół.

    Działanie:
    - Inicjalizuje węzeł ROS2 o nazwie 'robot_inverse_kinematics'.
    - Subskrybuje temat /cmd_vel i odbiera wiadomości typu Twist (zadane prędkości robota).
    - Tworzy publisher na temat '/cmd_joint_states', gdzie będą publikowane obliczone prędkości kół,
      a w zależności od kinematyki mogą być to również ich pozycje.
    """
    def __init__(self):
        # Inicjalizacja węzła ROS2 z unikalną nazwą 'robot_inverse_kinematics'.
        # Węzeł to podstawowy element w ROS2, który może subskrybować i publikować dane.
        super().__init__('robot_inverse_kinematics')
        
        # --- SUBSKRYPCJA prędkości sterujących robotem ---
        # Tworzymy subskrypcję na temat (topic) /cmd_vel. Ten temat standardowo służy w ROS do przekazywania 
        # zadanych prędkości robota (np. prędkość postępowa i obrotowa całego robota a nie pojedynczych kół).

        # Wiadomości na tym temacie są typu Twist i zawierają:
        #  - prędkości liniowe (x, y, z)
        #  - prędkości kątowe (wokół osi x, y, z)
        # Gdy odebrana zostanie nowa wiadomość (publisher wysyła nową wiadomość) to zostanie wywołana metoda
        # cmd_vel_callback().
        self.subscription_cmd_vel = self.create_subscription(
            Twist,                  # Typ wiadomości (wektor prędkości robota)
            '/cmd_vel',             # Nazwa tematu, z którego odbieramy dane.
            self.cmd_vel_callback,  # Funkcja wywoływana po odebraniu wiadomości
            10                      # QoS profile depth – ile wiadomości może buforować
        )

        
        # --- PUBLIKACJA wiadomości sterujących każdym z kół oddzielnie ---
        # Tworzymy publisher na temat /cmd_joint_states.
        # Na tym temacie będą publikowane wiadomości typu JointState, które opisują:
        #  - prędkości obrotowe kół (obliczone z kinematyki odwrotnej),
        #  - opcjonalnie ich aktualne pozycje (zależy od kinematyki robota).
        # Dzięki temu inne węzły (np. symulator, sterownik niskopoziomowy) mogą korzystać z obliczonych 
        # prędkości kół.
        
        self.publisher_joint_states = self.create_publisher(
            JointState,             # Typ wiadomości opisującej stan przegubów (kół)
            '/cmd_joint_states',    # Nazwa topicu publikacji
            10                      # QoS profile depth
        )

In [ ]:
# rclpy.ok() sprawdza, czy biblioteka rclpy jest już uruchomiona (czy działa "event loop").
# Jeżeli nie, wywołujemy rclpy.init(args=None), aby zainicjalizować środowisko ROS2 dla języka Python.
# Dzięki temu węzeł będzie mógł tworzyć subskrypcje, publishery i wykonywać callbacki.
if not rclpy.ok():
    rclpy.init(args=None)

---

<p style="text-align:center; font-size:24px;">Kinematyka odwrotna</p>
<img src="../images/odwrotna.png" alt="System architecture" width="800"/>


## Złożenie ograniczeń wszystkich kół

Dla robota z czterema kołami skrętnymi możemy połączyć wszystkie ograniczenia w postaci macierzowej:

$$
\mathbf{A} \mathbf{\dot{\xi_{R}}} = \mathbf{B} \mathbf{\dot{\varphi}}
$$

Gdzie:
- **\( A \)** – macierz ograniczeń kinematycznych wynikających z warunku toczenia i braku poślizgu,
- **\( B \)** – macierz przekształcająca prędkości kół na prędkość robota.

---

## Macierze dla robota z napędem omni (swerve drive)

### **Macierz $(\mathbf{A})$ – Ograniczenia kinematyczne**
$$
\mathbf{A} =
\begin{bmatrix}
\sin(\alpha_1 + \beta_1 + \theta_1)  & -\cos(\alpha_1 + \beta_1 + \theta_1) & -l_1 \cos(\beta_1 + \theta_1) \\
\sin(\alpha_2 + \beta_2 + \theta_2)  & -\cos(\alpha_2 + \beta_2 + \theta_2) & -l_2 \cos(\beta_2 + \theta_2) \\
\sin(\alpha_3 + \beta_3 + \theta_3)  & -\cos(\alpha_3 + \beta_3 + \theta_3) & -l_3 \cos(\beta_3 + \theta_3) \\
\sin(\alpha_4 + \beta_4 + \theta_4)  & -\cos(\alpha_4 + \beta_4 + \theta_4) & -l_4 \cos(\beta_4 + \theta_4) \\
\cos(\alpha_1 + \beta_1 + \theta_1)  & \sin(\alpha_1 + \beta_1 + \theta_1)  & l_1 \sin(\beta_1 + \theta_1) \\
\cos(\alpha_2 + \beta_2 + \theta_2)  & \sin(\alpha_2 + \beta_2 + \theta_2)  & l_2 \sin(\beta_2 + \theta_2) \\
\cos(\alpha_3 + \beta_3 + \theta_3)  & \sin(\alpha_3 + \beta_3 + \theta_3)  & l_3 \sin(\beta_3 + \theta_3) \\
\cos(\alpha_4 + \beta_4 + \theta_4)  & \sin(\alpha_4 + \beta_4 + \theta_4)  & l_4 \sin(\beta_4 + \theta_4)
\end{bmatrix}
$$

Gdzie:
- \( $\alpha_n$ \) – orientacja koła \( n \) w układzie robota (kąt położenia koła względem środka robota, liczony od osi X robota),
- \( $\beta_n$ \) – kąt skrętu koła,
- \( $\theta_n$ \) – obliczony kąt skrętu koła \( n \),
- \( $l_n$ \) – odległość koła od środka układu robota.

---

### **Macierz $(\mathbf{B})$ – Transformacja prędkości kół**
$$
\mathbf{B} =
\begin{bmatrix}
r & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & r & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & r & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & r & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 0
\end{bmatrix}
$$

Gdzie:
- \( r \) – promień koła.

---


## Obliczanie prędkości kół  $\mathbf{\dot{\varphi}}$

Korzystając z równań kinematyki odwrotnej:

$$
\mathbf{\dot{\phi}} = (\mathbf{B}^T \mathbf{B})^{-1} \mathbf{B}^T \mathbf{A} \mathbf{\dot{\xi}_R}
$$


Ostatecznie otrzymujemy wektor prędkości kół.

---

##  Funkcja pomocniczna

In [ ]:
def get_steering_angle(x, y, beta, cmd_vel_x, cmd_vel_y, cmd_vel_ang):
    """
    Funkcja pomocnicza do wyznaczania docelowego kąta skrętu koła
    na podstawie zadanych prędkości i położenia koła względem środka robota.

    W robotach z kołami skrętnymi (np. typu Ackermanna, Omni - swerve-drive)
    każde koło może mieć inny kąt skrętu, zależny od jego położenia
    względem środka robota oraz od tego, jakie prędkości zostały zadane
    w wiadomości `/cmd_vel`. Funkcja ta pozwala wyznaczyć ten kąt
    dla pojedynczego koła.
    
    Funkcja uwzględnia różne przypadki ruchu:
    - brak ruchu: zwracany jest kąt 0,
    - obrót w miejscu: zwracany jest kąt -beta,
    - ruch postępowy: wyznaczany jest kierunek ruchu,
    - ruch złożony (translacja + rotacja): obliczany jest rzeczywisty kierunek ruchu koła,
      z uwzględnieniem położenia (`x`, `y`) i rotacji (`cmd_vel_ang`).

    Parametry:
        x (float): Pozycja koła względem środka robota w osi X [m].
        y (float): Pozycja koła względem środka robota w osi Y [m].
        beta (float): Kąt skrętu koła względem osi robota (rad).
        cmd_vel_x (float): Prędkość liniowa robota w osi X (m/s).
        cmd_vel_y (float): Prędkość liniowa robota w osi Y (m/s).
        cmd_vel_ang (float): Prędkość kątowa wokół osi Z (rad/s).

    Zwraca:
        float: Docelowy kąt skrętu koła w radianach.
    """
    if cmd_vel_x == 0 and cmd_vel_y == 0 and cmd_vel_ang == 0:
        # No motion
        return 0.0
    elif cmd_vel_x == 0 and cmd_vel_y == 0 and cmd_vel_ang != 0:
        # Pure rotation case
        beta += -math.pi if beta > (math.pi)/2 else 0
        beta +=  math.pi if beta < -(math.pi)/2 else 0
        return -beta
    elif cmd_vel_ang == 0:
        # Pure translation case
        angle = math.atan2(cmd_vel_y, cmd_vel_x)
        # Prevent abrupt flipping for backward motion
        if cmd_vel_x < 0:
            angle += -math.pi if angle > 0 else math.pi
        return angle
    else:
        # Combined translation + rotation (Ackermann adjustment)
        wheel_vel_x = cmd_vel_x - cmd_vel_ang * y
        wheel_vel_y = cmd_vel_y + cmd_vel_ang * x

        steering_angle = math.atan2(wheel_vel_y, wheel_vel_x)
        # Adjust for reverse motion
        if cmd_vel_x < 0:
            steering_angle += -math.pi if steering_angle > 0 else math.pi

        return steering_angle

## <span style="color:red">Uzupełnij poniższą komórkę. (2/2)</span>
Należy uzupełnić funkcję $\mathbf{cmd\_vel\_callback}$ aby realizowała funkcjonalność kinematyki odwrotnej robota mobilnego.
1.  (Opcjonalnie) Obliczyć kąty skrętu kół $\mathbf{\theta}$, jeśli kinematyka robota tego wymaga.
2.  Obliczyć prędkości kół $\mathbf{\dot{\phi}}$ na podstawie wektora prędkości robota $\mathbf{\dot{\xi}_R}$. Należy skorzystać ze wzoru:
	
	\begin{equation}
		\mathbf{\dot{\phi}} = (\mathbf{B}^T \mathbf{B})^{-1} \mathbf{B}^T \mathbf{A} \mathbf{\dot{\xi}_R}
	\end{equation}
	gdzie $\mathbf{A}$ jest macierzą ograniczeń kinematycznych a $\mathbf{B}$ jest macierzą transformacji prędkości kół. 
3. Uzupełnić wiadomość $\mathbf{joint\_state}$.
4. Zweryfikować poprawność wyników na rzeczywistym robocie. Sprawdzić czy po wysłaniu sterowania z interfejsu użytkownika (Jupyter Notebook) robot porusza się w odpowiednim kierunku np. jeśli ma jechać do przodu to jedzie do przodu a nie do tyłu albo wcale się nie porusza. Wystarczą testy polegające na obserwacji zachowania robota po zadaniu sterowania. Bez szczegółowych testów i zbierania danych.

In [ ]:
def cmd_vel_callback(self, msg):
    """
    Callback odbierający wiadomość typu Twist z tematu /cmd_vel. Wiadomość zawiera zadany wektor prędkości robota
    na podstawie którego obliczane są prędkości oraz kąty skrętu kół. Na podstawie odebranej wiadomości 
    należy obliczyć prędkości (i ewentualnie kąty skrętu) kół oraz opublikować wynik w wiadomości typu JointState.

    :param msg: Wiadomość typu geometry_msgs.msg.Twist zawierająca:
        - msg.linear.x (float): prędkość liniowa robota wzdłuż osi X (m/s)
        - msg.linear.y (float): prędkość liniowa robota wzdłuż osi Y (m/s) – zwykle 0 dla robotów różnicowych
        - msg.angular.z (float): prędkość kątowa wokół osi Z (rad/s), odpowiadająca skręcaniu robota

    Pożądane wyjście:
    - Wysłanie wiadomości typu sensor_msgs.msg.JointState na temat /cmd_joint_states, gdzie:
        - joint_state.velocity (list(float)) - Prędkości obrotowe kół [rad/s], w kolejności 
                                               [wheel0, wheel1, wheel2, …].
        - joint_state.position (list[float]) - Kąty skrętu kół [rad], w tej samej kolejności 
                                               [wheel0, wheel1, wheel2, …].
    """   
    # 1. UZUPEŁNIJ [opcjonalnie]: jeśli potrzebne, oblicz kąty skrętu kół (position)
    # Aby stwierdzić czy należy uzupełnić to pole należy zastanowić się czy możemy w jakiś sposób wpłynąć na
    # ustawienie kątu skrętu koła. Jeśli tak to należy wyznaczyć wartości kątów skrętu kół, ale tylko dla kół, 
    # które tego wymagają. Wektor pozycji kół należy podać w tej samej kolejności co wektor prędkości kół. 
    # Dla kół bez kąta skrętu należy wpisać 0.0. Długość tego wektora ma być zgodna z długością wektora dla velocity.
    
    
    # 2. UZUPEŁNIJ: oblicz prędkości obrotowe kół (velocity) na podstawie wektora prędkości robota

    
    joint_state = JointState()
    # 3. UZUPEŁNIJ: joint_state zgodnie z dokumentacją “Pożądanego wyjścia”.


    # Publikacja wiadomości typu JointState na temacie /cmd_joint_states
    # self.publisher_joint_states - publisher zdefiniowany w konstruktorze klasy Robot, który zawiera metodę
    # publish. Metoda ta jako argument przymuje obiekt (wiadomość) dla ROS2. Typ wiadomości zależy od argumentów
    # przekazanych w konstruktorze. W tym przypadku przekazany obiekt joint_state jest typu JointState.
    self.publisher_joint_states.publish(joint_state)

In [ ]:
# Robot.cmd_vel_callback = cmd_vel_callback
#
# Ta konstrukcja przypisuje wcześniej zdefiniowaną funkcję `cmd_vel_callback`
# do klasy `Robot` jako jej metodę.
#
# Dzięki temu obiekty klasy Robot mogą wywoływać tę funkcję tak,
# jakby była napisana wewnątrz klasy:
#     robot = Robot()
#     robot.cmd_vel_callback(msg)
#
# Jest to sposób „doklejenia” funkcji do klasy po jej zdefiniowaniu.
Robot.cmd_vel_callback = cmd_vel_callback

In [ ]:
def main(args=None):
    # Tworzy instancję noda = uruchamia węzeł ROS2 (zdefiniowany w klasie Robot)
    # W tym momencie dostępne są już subskrypcje i publishery.
    node = Robot()

    # Główna pętla programu: dopóki system ROS2 działa,
    # obsługiwane są wszystkie przychodzące wiadomości i callbacki.
    # spin_once() wykonuje jedną iterację pętli zdarzeń, w odróżnieniu od spin(),
    # który działa w trybie ciągłym i blokującym.
    while rclpy.ok():
        rclpy.spin_once(node)

    # Można opcjonalnie usunąć noda ręcznie:
    # node.destroy_node()

    try:
        # Alternatywny sposób: rclpy.spin(node) – zwykle prostszy, ale mniej elastyczny.
        pass
    finally:
        # Bezpieczne zamknięcie ROS2 nawet przy błędach lub przerwaniu programu.
        if rclpy.ok():
            rclpy.shutdown()

# Standardowy idiom w Pythonie: uruchomienie main() tylko wtedy,
# gdy ten plik jest wykonywany bezpośrednio (a nie importowany).
# Uruchomienie funkcji głównej po uruchomieniu skryptu
if __name__ == '__main__':
    main()